# PAD-UFES-20 Image-Only Baseline

Run this notebook in Google Colab with a GPU runtime. It trains an image-only baseline for clinician triage support, using patient-safe splits generated by `src.data.make_image_splits`.

Raw PAD-UFES-20 data is downloaded from the public Hugging Face dataset mirror:

```text
SalmaneExploring/pad-ufes-20
```

MLflow logs to the project DagsHub tracking server. Store `DAGSHUB_TOKEN` in Colab Secrets before running; set `DAGSHUB_USERNAME`, `DAGSHUB_REPO_OWNER`, or `DAGSHUB_MLFLOW_TRACKING_URI` too if they differ from the defaults. Google Drive is still used for checkpoint and report backups.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')
DATA_ROOT = Path('/content/pad_ufes_20')
IMAGES_DIR = DATA_ROOT / 'all_images'
OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/image_baseline')
DAGSHUB_REPO_OWNER = get_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
DAGSHUB_REPO_NAME = get_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
DAGSHUB_USERNAME = get_config('DAGSHUB_USERNAME', DAGSHUB_REPO_OWNER)
DAGSHUB_MLFLOW_TRACKING_URI = get_config(
    'DAGSHUB_MLFLOW_TRACKING_URI',
    f'https://dagshub.com/{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}.mlflow',
)
DAGSHUB_TOKEN = get_config('DAGSHUB_TOKEN')
MLFLOW_EXPERIMENT_NAME = 'pad-ufes-20-image-baseline'

if not DAGSHUB_TOKEN:
    raise RuntimeError(
        'Set DAGSHUB_TOKEN in Colab Secrets before training so MLflow can log to DagsHub.'
    )

os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def repo_url_with_optional_token(repo_url):
    token = get_config('GH_TOKEN')
    if not token:
        return repo_url
    return repo_url.replace('https://', f'https://x-access-token:{token}@', 1)


AUTH_REPO_URL = repo_url_with_optional_token(REPO_URL)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, AUTH_REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('Hugging Face dataset:', HF_DATASET_REPO)
print('PAD-UFES-20 data root:', DATA_ROOT)
print('PAD-UFES-20 images dir:', IMAGES_DIR)
print('MLflow tracking URI:', DAGSHUB_MLFLOW_TRACKING_URI)


Working directory: /content/mlops-teledermatology
Hugging Face dataset: SalmaneExploring/pad-ufes-20
PAD-UFES-20 data root: /content/pad_ufes_20
PAD-UFES-20 images dir: /content/pad_ufes_20/all_images
MLflow tracking URI: https://dagshub.com/SalmaneSossey/mlops-teledermatology.mlflow


In [3]:
!pip -q install mlflow huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.2/866.2 kB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!python -m src.data.download_pad_ufes_20 \
  --repo-id "{HF_DATASET_REPO}" \
  --output-dir "{DATA_ROOT}" \
  --force

!python -m src.data.make_image_splits \
  --metadata-path "{DATA_ROOT / 'metadata.csv'}" \
  --images-dir "{IMAGES_DIR}" \
  --output-dir data/processed/splits


Fetching ... files: 2301it [05:45,  6.67it/s]
Download complete: : 3.57GB [06:00, 4.21MB/s]Downloaded SalmaneExploring/pad-ufes-20 to /content/pad_ufes_20
Metadata: /content/pad_ufes_20/metadata.csv
Images: /content/pad_ufes_20/all_images
Download complete: : 3.57GB [06:09, 9.67MB/s]
Split distribution by diagnosis:
split       test  train  val
diagnostic                  
ACK          109    511  110
BCC          127    591  127
MEL            8     36    8
NEV           36    171   37
SCC           29    134   29
SEK           35    165   35

Images per split:
split
train    1608
val       346
test      344
Name: count, dtype: int64

Patients per split:
split
train    959
val      208
test     206
Name: patient_id, dtype: int64

Wrote split manifests to data/processed/splits


In [5]:
import json
import random
import shutil
import time

import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8
NUM_WORKERS = 2
REQUIRE_GPU = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and device.type != 'cuda':
    raise RuntimeError('No GPU detected. In Colab, choose Runtime > Change runtime type > T4 GPU, then rerun the notebook.')
device

device(type='cuda')

In [6]:
SPLIT_DIR = Path('data/processed/splits')

train_df = pd.read_csv(SPLIT_DIR / 'train.csv')
val_df = pd.read_csv(SPLIT_DIR / 'val.csv')
test_df = pd.read_csv(SPLIT_DIR / 'test.csv')

with open(SPLIT_DIR / 'label_mapping.json') as f:
    label_mapping = json.load(f)
with open(SPLIT_DIR / 'class_weights.json') as f:
    class_weight_payload = json.load(f)
with open(SPLIT_DIR / 'preprocessing_summary.json') as f:
    preprocessing_summary = json.load(f)

index_to_label = {int(k): v for k, v in label_mapping['index_to_label'].items()}
labels = [index_to_label[i] for i in range(len(index_to_label))]
high_risk_indices = [labels.index(label) for label in ['BCC', 'MEL', 'SCC']]

display(pd.crosstab(train_df['diagnostic'], train_df['split']))
print('Labels:', labels)
print('Train rows:', len(train_df), 'Val rows:', len(val_df), 'Test rows:', len(test_df))

split,train
diagnostic,
ACK,511
BCC,591
MEL,36
NEV,171
SCC,134
SEK,165


Labels: ['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']
Train rows: 1608 Val rows: 346 Test rows: 344


In [7]:
class PadUfesImageDataset(Dataset):
    def __init__(self, frame, images_dir, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_path = self.images_dir / row['image_rel_path']
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['label_idx'])
        return image, label


weights = models.EfficientNet_B0_Weights.DEFAULT
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_loader = DataLoader(
    PadUfesImageDataset(train_df, IMAGES_DIR, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(PadUfesImageDataset(val_df, IMAGES_DIR, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(PadUfesImageDataset(test_df, IMAGES_DIR, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [8]:
def build_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


model = build_model(num_classes=len(labels)).to(device)
class_weights = torch.tensor([class_weight_payload['class_weights'][label] for label in labels], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
amp_enabled = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

mlflow.set_tracking_uri(DAGSHUB_MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
if mlflow.active_run() is not None:
    mlflow.end_run()
run = mlflow.start_run(run_name=f'efficientnet_b0_{time.strftime("%Y%m%d_%H%M%S")}')
mlflow.set_tags({
    'model_family': 'cnn',
    'backbone': 'efficientnet_b0',
    'task': 'pad_ufes_20_image_classification',
    'triage_goal': 'prioritize_high_risk_dermatology_cases',
    'tracking_backend': 'dagshub_mlflow',
})
mlflow.log_params({
    'seed': SEED,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'num_workers': NUM_WORKERS,
    'optimizer': 'AdamW',
    'learning_rate': 3e-4,
    'weight_decay': 1e-4,
    'scheduler': 'CosineAnnealingLR',
    'loss': 'weighted_cross_entropy',
    'device': str(device),
    'hf_dataset_repo': HF_DATASET_REPO,
    'train_rows': len(train_df),
    'val_rows': len(val_df),
    'test_rows': len(test_df),
})
mlflow.log_dict(label_mapping, 'label_mapping.json')
mlflow.log_dict(class_weight_payload, 'class_weights.json')
mlflow.log_dict(preprocessing_summary, 'preprocessing_summary.json')


sum(p.numel() for p in model.parameters())

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 115MB/s] 


MlflowException: API request to endpoint /api/2.0/mlflow/experiments/get-by-name failed with error code 401 != 200. Response body: '=============== ATTENTION! ===============

To use authentication, you must first: 
    Get your default access token from: https://dagshub.com/user/settings/tokens
    OR 
    Set a password: https://dagshub.com/user/settings/password 
=========================================='

In [ ]:
def run_epoch(model, loader, train=False):
    model.train(train)
    running_loss = 0.0
    all_targets = []
    all_preds = []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(images)
                loss = criterion(logits, targets)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * images.size(0)
        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    epoch_loss = float(running_loss / len(loader.dataset))
    macro_f1 = float(f1_score(all_targets, all_preds, average='macro', zero_division=0))
    balanced_acc = float(balanced_accuracy_score(all_targets, all_preds))
    high_mask = np.isin(all_targets, high_risk_indices)
    high_recall = float(recall_score(high_mask, np.isin(all_preds, high_risk_indices), zero_division=0))
    return {'loss': epoch_loss, 'macro_f1': macro_f1, 'balanced_acc': balanced_acc, 'high_risk_recall': high_recall}


best_metric = -1.0
best_path = OUTPUT_DIR / 'efficientnet_b0_best.pt'
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_metrics = run_epoch(model, train_loader, train=True)
    val_metrics = run_epoch(model, val_loader, train=False)
    scheduler.step()

    score = float(0.5 * val_metrics['macro_f1'] + 0.5 * val_metrics['high_risk_recall'])
    if score > best_metric:
        best_metric = score
        torch.save({'model_state_dict': model.state_dict(), 'labels': labels, 'epoch': epoch, 'val_metrics': val_metrics}, best_path)

    row = {'epoch': epoch, 'seconds': round(time.time() - start, 1), **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)
    mlflow.log_metrics({**row, 'selection_score': score, 'learning_rate': scheduler.get_last_lr()[0]}, step=epoch)
    print(row)

history_path = OUTPUT_DIR / 'history.csv'
pd.DataFrame(history).to_csv(history_path, index=False)
mlflow.log_artifact(history_path)
best_path

In [ ]:
# This checkpoint is written by this notebook. weights_only=False keeps
# PyTorch 2.6+ able to load earlier runs that saved NumPy metric scalars.
checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model = build_model(num_classes=len(labels)).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_targets = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_targets.extend(targets.numpy().tolist())
        all_preds.extend(probs.argmax(axis=1).tolist())

all_probs = np.concatenate(all_probs)
report = classification_report(all_targets, all_preds, target_names=labels, zero_division=0, output_dict=True)
cm = confusion_matrix(all_targets, all_preds, labels=list(range(len(labels))))
high_target = np.isin(all_targets, high_risk_indices)
high_pred = np.isin(all_preds, high_risk_indices)

test_metric_values = {
    'test_macro_f1': float(f1_score(all_targets, all_preds, average='macro', zero_division=0)),
    'test_balanced_accuracy': float(balanced_accuracy_score(all_targets, all_preds)),
    'test_high_risk_recall': float(recall_score(high_target, high_pred, zero_division=0)),
}
metrics = {
    **test_metric_values,
    'best_checkpoint': str(best_path),
    'labels': labels,
}

test_metrics_path = OUTPUT_DIR / 'test_metrics.json'
classification_report_path = OUTPUT_DIR / 'classification_report.csv'
confusion_matrix_path = OUTPUT_DIR / 'confusion_matrix.csv'

test_metrics_path.write_text(json.dumps(metrics, indent=2))
pd.DataFrame(report).T.to_csv(classification_report_path)
pd.DataFrame(cm, index=labels, columns=labels).to_csv(confusion_matrix_path)

mlflow.log_metrics(test_metric_values)
mlflow.log_artifact(test_metrics_path)
mlflow.log_artifact(classification_report_path)
mlflow.log_artifact(confusion_matrix_path)
mlflow.log_artifact(best_path)
mlflow.pytorch.log_model(model, artifact_path='model')
mlflow.end_run()

metrics